In [1]:
# import libraries
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px

In [2]:
# load in dataframes
salt2 = pd.read_csv('Salt_Intake2.csv')
obesity = pd.read_csv('Obesity.csv')
colon2 = pd.read_csv('colon_cancer_incidence.csv')

FileNotFoundError: [Errno 2] No such file or directory: 'Salt_Intake.csv'

In [ ]:
## Data Preparation to make a combined dataframe

# map country names to the correct form
country_mapping = {
    'USA': 'United States of America',
    'Bahrain: Bahrainis': 'Bahrain',
    'Iran (Islamic Republic of)': 'Iran',
    'Republic of Korea': 'South Korea',
    'UK': "United Kingdom",
    'Russian Federation': 'Russia',
    'United Kingdom of Great Britain and Northern Ireland': 'United Kingdom',
    'The Netherlands': 'Netherlands'
}


# Filter for relevant rows in salt intake dataset
salt_filtered = salt2[salt2['Indicator Name'] == 'Mean population salt intake, adults aged 25+']
salt_filtered = salt_filtered[['Country Name', 'Gender', 'Numeric']].rename(columns={'Country Name': 'Country', 'Numeric': 'Salt_Intake'})
salt_filtered = salt_filtered[salt_filtered['Gender'] == "Total"]

# Get the right name for the country
salt_filtered['Country'] = salt_filtered['Country'].str.split(',', n=1).str[0].str.strip()
salt_filtered['Country'] = salt_filtered['Country'].map(country_mapping).fillna(salt_filtered['Country'])

# Clean and aggregate the colon cancer data
# Extract country names by splitting at commas and taking the first part
colon2 = colon2.rename(columns={'Location': 'Country'})
colon2['Country'] = colon2['Country'].str.split(',').str[0]

# slice and group dataframe
colon2_sliced = colon2[['Country', 'TOTAL']]
colon2_sliced_grouped = colon2_sliced.groupby('Country', as_index=False).sum().rename(columns={'TOTAL': 'Colon_Cancer_Rate'})

# remove unnecessary rows
colon2_sliced_grouped = colon2_sliced_grouped[colon2_sliced_grouped['Country'] != 'USA: Black']
colon2_sliced_grouped = colon2_sliced_grouped[colon2_sliced_grouped['Country'] != 'USA: White']
colon2_sliced_grouped = colon2_sliced_grouped[colon2_sliced_grouped['Country'] != 'Israel: Arabs']
colon2_sliced_grouped = colon2_sliced_grouped[colon2_sliced_grouped['Country'] != 'Israel: Jews']
colon2_sliced_grouped = colon2_sliced_grouped[colon2_sliced_grouped['Country'] != 'Kuwait: Non-Kuwaitis']
colon2_sliced_grouped = colon2_sliced_grouped[colon2_sliced_grouped['Country'] != 'Kuwait: Kuwaitis']
colon2_sliced_grouped = colon2_sliced_grouped[colon2_sliced_grouped['Country'] != 'New Zealand: Maori']
colon2_sliced_grouped = colon2_sliced_grouped[colon2_sliced_grouped['Country'] != 'New Zealand: Other']
colon2_sliced_grouped = colon2_sliced_grouped[colon2_sliced_grouped['Country'] != 'New Zealand: Pacific peoples']
colon2_sliced_grouped = colon2_sliced_grouped[colon2_sliced_grouped['Country'] != 'Singapore: Chinese']
colon2_sliced_grouped = colon2_sliced_grouped[colon2_sliced_grouped['Country'] != 'Singapore: Indian']
colon2_sliced_grouped = colon2_sliced_grouped[colon2_sliced_grouped['Country'] != 'Singapore: Malay']
colon2_sliced_grouped['Country'] = colon2_sliced_grouped['Country'].map(country_mapping).fillna(colon2_sliced_grouped['Country'])

# Merge datasets on the 'Country' column
combined_df = pd.merge(salt_filtered, colon2_sliced_grouped, on='Country', how='inner')


In [ ]:
combined_df.to_csv('merged_salt_cancer6.csv')

In [ ]:
import pandas as pd
import folium
import json
import requests
from branca.colormap import LinearColormap
from IPython.display import HTML, display

# Read the CSV file and make sure values are numeric
data = pd.read_csv('merged_salt_cancer6.csv')
data['Salt_Intake'] = pd.to_numeric(data['Salt_Intake'], errors='coerce')
data['Colon_Cancer_Rate'] = pd.to_numeric(data['Colon_Cancer_Rate'], errors='coerce')

# Function to display Folium map in Jupyter
def display_map(map_object):
    """Display a folium map in Jupyter notebook."""
    return HTML(map_object._repr_html_())

# Create a style function for hover effect
style_function = lambda x: {'fillColor': 'transparent', 'color': 'transparent'}

# Download GeoJSON data for world countries
url = "https://raw.githubusercontent.com/python-visualization/folium/master/examples/data/world-countries.json"
geojson_data = requests.get(url).json()

# Create a lookup dictionary for salt intake values
salt_lookup = dict(zip(data['Country'], data['Salt_Intake']))
cancer_lookup = dict(zip(data['Country'], data['Colon_Cancer_Rate']))

# Add salt intake and cancer rate data to GeoJSON properties
for feature in geojson_data['features']:
    country_name = feature['properties']['name']
    feature['properties']['salt_intake'] = salt_lookup.get(country_name, 'No data')
    feature['properties']['cancer_rate'] = cancer_lookup.get(country_name, 'No data')

# Create the first map for Salt Intake
m1 = folium.Map(location=[20, 0], zoom_start=2, height='600px')

# Add the chloropleth layer for salt intake
choropleth1 = folium.Choropleth(
    geo_data=geojson_data,
    name='choropleth',
    data=data,
    columns=['Country', 'Salt_Intake'],
    key_on='feature.properties.name',
    fill_color='YlOrRd',
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name='Salt Intake (g/day)',
    highlight=True
).add_to(m1)

# Add tooltips for salt intake with actual values
folium.GeoJson(
    geojson_data,
    style_function=style_function,
    tooltip=folium.GeoJsonTooltip(
        fields=['name', 'salt_intake'],
        aliases=['Country: ', 'Salt Intake (g/day): '],
        localize=True,
        sticky=True,
        labels=True
    )
).add_to(m1)

# Display the first map
display(HTML("<h2>Daily Salt Intake by Country</h2>"))
display(display_map(m1))

# Create the second map for Cancer Rates
m2 = folium.Map(location=[20, 0], zoom_start=2, height='600px')

# Add the chloropleth layer for cancer rates
choropleth2 = folium.Choropleth(
    geo_data=geojson_data,
    name='choropleth',
    data=data,
    columns=['Country', 'Colon_Cancer_Rate'],
    key_on='feature.properties.name',
    fill_color='Blues',
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name='Colon Cancer Incidents',
    highlight=True
).add_to(m2)

# Add tooltips for cancer rates with actual values
folium.GeoJson(
    geojson_data,
    style_function=style_function,
    tooltip=folium.GeoJsonTooltip(
        fields=['name', 'cancer_rate'],
        aliases=['Country: ', 'Total Cancer Incidents: '],
        localize=True,
        sticky=True,
        labels=True
    )
).add_to(m2)

# Display the second map
display(HTML("<h2>Colon Cancer Incidence by Country (2013-2017)</h2>"))
display(display_map(m2))